<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/01_tabular/01_classification_quest_titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚢 Quest · Tabular classification: Would you have survived the Titanic?

**April 15, 1912, 2:20 a.m.** The RMS *Titanic*, "the unsinkable ship", disappears into the North Atlantic after hitting
an iceberg. Of the ~2 200 people aboard, more than 1 500 die. There were lifeboats for barely half of them, and the
order on deck was *"women and children first"*… but was it really applied? And did wealth matter?

A century later, in 2012, Kaggle turned the passenger list into its most famous tutorial competition. Millions of
data scientists wrote their first model on it. Today it's your turn.

**Your mission:** predict whether a passenger survived (`1`) or not (`0`), a **binary classification** task.
**Metric:** accuracy (share of correct predictions), like on Kaggle.

| You will practise | Concept |
|---|---|
| Loading & exploring a dataset | EDA, target rate, groupby |
| Spotting **data leakage** | features that "know" the answer |
| Creating features from text | regex, domain knowledge |
| Preprocessing without leakage | `Pipeline`, `ColumnTransformer` |
| Comparing models honestly | baselines, cross-validation, leaderboard |
| Producing a submission | refit on all data, predict, save |

🧩 Cells marked `# TODO` are yours. Replace the `...`. Run the `check(...)` cells to know if you are on track.
💡 Stuck for more than 10 minutes? Reveal the solution below the cell, understand it, and move on.

In [ ]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

quest("Would you have survived the Titanic?", chapter="Track 1 · Tabular · Classification", xp=300)

## 1. Load the passenger list 📥

The data comes from [OpenML](https://www.openml.org/d/40945) (an open repository of datasets for ML): 1 309
passengers with their ticket class, name, sex, age, family aboard, fare, cabin, port of embarkation… and fate.

We keep **900 passengers** as our training set and hide the remaining **409** in a "test set", exactly like Kaggle does
(Kaggle's train has 891 rows). We will only look at the test set at the very end.

In [ ]:
df = openml(40945)
df["survived"] = df["survived"].astype(int)

train = df.sample(n=900, random_state=SEED)
test = df.drop(train.index)
test_labels = test.pop("survived")             # hidden answers: the "private leaderboard"
print(train.shape, test.shape)
train.head()

### 🎯 Exercise 1: what share of passengers survived in `train`?
This number is crucial: it is the accuracy of the dumbest possible model.

In [ ]:
survival_rate = ...   # TODO: a float between 0 and 1 (hint: the mean of a 0/1 column)
print(f"Survival rate: {survival_rate:.1%}")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
survival_rate = train["survived"].mean()
print(f"Survival rate: {survival_rate:.1%}")
```

</details>

In [ ]:
check(isinstance(survival_rate, float) and 0.35 < survival_rate < 0.42,
      "Correct! Only ~38% survived. A model predicting 'died' for everyone already gets ~62% accuracy.",
      "Use train['survived'].mean().")

## 2. Explore: women and children first? 🔍

EDA is a conversation with the data: ask a question, make a chart, write the answer.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=train, x="sex", y="survived", ax=axes[0]).set(title="Survival rate by sex")
sns.barplot(data=train, x="pclass", y="survived", ax=axes[1]).set(title="Survival rate by class")
sns.histplot(data=train, x="age", hue="survived", bins=30, multiple="stack", ax=axes[2]).set(title="Age distribution")
plt.tight_layout(); plt.show()

### 🎯 Exercise 2: survival rate by sex **and** class
Build a table with `sex` as rows, `pclass` as columns and the survival rate in each cell.

In [ ]:
table = ...   # TODO: hint: train.pivot_table(values=..., index=..., columns=..., aggfunc=...)
table

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
table = train.pivot_table(values="survived", index="sex", columns="pclass", aggfunc="mean", observed=True)
table
```

</details>

In [ ]:
check(isinstance(table, pd.DataFrame) and table.shape == (2, 3) and table.loc["female", 1] > 0.85,
      "Great! A 1st-class woman had >90% chance to survive; a 3rd-class man, about 15%. Class mattered a lot.",
      "Expected a 2×3 table: index = sex, columns = pclass, values = mean of survived.")

**Conclusions so far** (always write them!): sex is the strongest signal, then class. Children seem to survive more.
These ideas will guide our features.

## 3. 🕵️ The leakage hunt

Some columns were filled **after** the sinking. A model using them would get a perfect score… and be useless
(and you'd fall to the bottom of a real leaderboard, where these columns would be absent or the trick banned).

Look at the description of the columns:
- `boat`: lifeboat number (if the passenger was in one)
- `body`: body identification number (if the body was recovered)
- `home.dest`: home / destination
- `ticket`, `cabin`: ticket number, cabin number

### 🎯 Exercise 3: which columns leak the answer?

In [ ]:
print(train.groupby(train["boat"].notna())["survived"].mean().rename("survival rate if boat is known"))
print(train.groupby(train["body"].notna())["survived"].mean().rename("survival rate if body is known"))

In [ ]:
leaky_columns = [...]   # TODO: list the names of the leaking columns

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
leaky_columns = ["boat", "body"]
```

</details>

In [ ]:
check(set(leaky_columns) == {"boat", "body"},
      "Exactly. Being in a lifeboat ⇒ survived; having a body number ⇒ died. Drop them!",
      "Two columns are filled after the sinking. Look at the two survival rates printed above.")
train = train.drop(columns=["boat", "body"]); test = test.drop(columns=["boat", "body"])
checkpoint("titanic/leakage", xp=50, msg="Leakage hunter!")

## 4. 🛠️ Feature engineering: reading between the lines

Names look useless… but they contain **titles**: *Mr*, *Mrs*, *Miss*, *Master* (a boy), *Dr*, *Rev*, *Countess*…
A title encodes sex, age and social status at once.

Other classic ideas: the **family size** (alone vs big families), the **deck** (first letter of the cabin: decks
A-C were close to the lifeboats), and whether the cabin is known at all.

### 🎯 Exercise 4: write `make_features`
Complete the function so it adds:
- `title`: extracted from the name (`"Allen, Miss. Elisabeth"` → `"Miss"`), rare titles grouped into `"Rare"`
- `family_size` = `sibsp` + `parch` + 1
- `is_alone` = 1 if `family_size == 1` else 0
- `deck`: first letter of `cabin`, `"U"` (unknown) if missing

In [ ]:
def make_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    # Title: the word between ", " and "."  (hint: data["name"].str.extract(r",\s*([^\.]+)\."))
    data["title"] = ...
    data["title"] = data["title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    data.loc[~data["title"].isin(["Mr", "Mrs", "Miss", "Master"]), "title"] = "Rare"
    data["family_size"] = ...
    data["is_alone"] = ...
    data["deck"] = ...     # hint: .str[0] and .fillna("U")
    return data

train_fe, test_fe = make_features(train), make_features(test)
train_fe[["name", "title", "family_size", "is_alone", "deck"]].head()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def make_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data["title"] = data["name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    data["title"] = data["title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    data.loc[~data["title"].isin(["Mr", "Mrs", "Miss", "Master"]), "title"] = "Rare"
    data["family_size"] = data["sibsp"] + data["parch"] + 1
    data["is_alone"] = (data["family_size"] == 1).astype(int)
    data["deck"] = data["cabin"].str[0].fillna("U")
    return data

train_fe, test_fe = make_features(train), make_features(test)
train_fe[["name", "title", "family_size", "is_alone", "deck"]].head()
```

</details>

In [ ]:
check(set(train_fe["title"].unique()) <= {"Mr", "Mrs", "Miss", "Master", "Rare"}
      and train_fe["family_size"].min() == 1 and set(train_fe["is_alone"].unique()) == {0, 1}
      and train_fe["deck"].isna().sum() == 0,
      "Features ready! Look at the survival rate of 'Master' (boys) below: children first indeed.",
      "Check each new column: titles in {Mr, Mrs, Miss, Master, Rare}, family_size >= 1, deck without NaN.")
train_fe.groupby("title")["survived"].agg(["mean", "count"]).round(2)

## 5. ⚙️ Preprocessing inside a Pipeline

Models need numbers without holes. We will:
- **impute** missing ages/fares with the median, then **scale** numbers,
- **impute** missing categories with the most frequent value, then **one-hot encode** them.

Putting this in a `ColumnTransformer` inside a `Pipeline` guarantees that the medians and categories are learnt on
training folds only (no leakage, remember the playbook?) and gives one object to export.

### 🎯 Exercise 5: build the preprocessor

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = ["age", "fare", "sibsp", "parch", "family_size", "is_alone", "pclass"]
cat_cols = ["sex", "embarked", "title", "deck"]
X, y = train_fe[num_cols + cat_cols], train_fe["survived"]
X_test = test_fe[num_cols + cat_cols]

In [ ]:
preprocess = ColumnTransformer([
    ("num", make_pipeline(...), num_cols),   # TODO: SimpleImputer(strategy="median") then StandardScaler()
    ("cat", make_pipeline(...), cat_cols),   # TODO: SimpleImputer(strategy="most_frequent") then OneHotEncoder(handle_unknown="ignore")
])

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat_cols),
])
```

</details>

In [ ]:
Xt = preprocess.fit_transform(X)
check(Xt.shape[0] == 900 and not np.isnan(Xt.toarray() if hasattr(Xt, "toarray") else Xt).any(),
      f"Preprocessing works: {Xt.shape[1]} numeric columns, no missing values.",
      "The transformed matrix should have 900 rows and no NaN.")

## 6. 🧱 Baselines, then real models

We compare every model with **5-fold stratified cross-validation** and log the results in a leaderboard.

Two baselines first:
1. **Dummy**: always predicts the majority class ("died").
2. **The 1912 rule**: "women survive, men die". This is what the crew tried to apply. Can ML do better than a
   one-line rule?

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
lb = Leaderboard("titanic", metric="accuracy")

lb.add("Dummy (everyone dies)", cross_val_score(DummyClassifier(), X, y, cv=cv).mean())
lb.add("1912 rule (women survive)", ((X["sex"] == "female").astype(int) == y).mean())

### 🎯 Exercise 6: cross-validate three real models
Wrap each model in a pipeline with `preprocess`, compute the 5-fold CV accuracy and add it to the leaderboard.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED),
    "Gradient Boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, random_state=SEED),
}
for name, model in models.items():
    pipe = ...                      # TODO: make_pipeline(preprocess, model)
    scores = ...                    # TODO: cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
    lb.add(name, scores.mean(), std=round(scores.std(), 3))
lb.show()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED),
    "Gradient Boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, random_state=SEED),
}
for name, model in models.items():
    pipe = make_pipeline(preprocess, model)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
    lb.add(name, scores.mean(), std=round(scores.std(), 3))
lb.show()
```

</details>

In [ ]:
check(len(lb.rows) >= 5 and lb.best()["accuracy"] > 0.80,
      "Your models beat the 1912 rule. Note the 'std' column: differences smaller than it are not significant!",
      "Each model should reach ~0.80+. Did you use make_pipeline(preprocess, model)?")
checkpoint("titanic/models", xp=80)

## 7. 🎛️ Tuning: find better hyperparameters

Hyperparameters are the knobs of a model (number of trees, depth…). `RandomizedSearchCV` tries random combinations
and keeps the best one according to cross-validation.

### 🎯 Exercise 7: tune the random forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

pipe = make_pipeline(preprocess, RandomForestClassifier(random_state=SEED))
param_distributions = {                       # parameters of a pipeline step are named <step>__<param>
    "randomforestclassifier__n_estimators": [200, 400, 600],
    "randomforestclassifier__max_depth": [4, 6, 8, None],
    "randomforestclassifier__min_samples_leaf": [1, 2, 4, 8],
    "randomforestclassifier__max_features": ["sqrt", 0.5],
}
search = ...   # TODO: RandomizedSearchCV(pipe, param_distributions, n_iter=15, cv=cv, scoring="accuracy", random_state=SEED, n_jobs=-1)
search.fit(X, y)
print(search.best_params_)
lb.add("Random Forest (tuned)", search.best_score_)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from sklearn.model_selection import RandomizedSearchCV

pipe = make_pipeline(preprocess, RandomForestClassifier(random_state=SEED))
param_distributions = {
    "randomforestclassifier__n_estimators": [200, 400, 600],
    "randomforestclassifier__max_depth": [4, 6, 8, None],
    "randomforestclassifier__min_samples_leaf": [1, 2, 4, 8],
    "randomforestclassifier__max_features": ["sqrt", 0.5],
}
search = RandomizedSearchCV(pipe, param_distributions, n_iter=15, cv=cv, scoring="accuracy",
                            random_state=SEED, n_jobs=-1)
search.fit(X, y)
print(search.best_params_)
lb.add("Random Forest (tuned)", search.best_score_)
```

</details>

## 8. 📤 Final model, test set and submission

Time to face the hidden passengers. We refit the best pipeline on **all** 900 training passengers, predict the 409
test passengers, save a submission file and reveal our "private leaderboard" score.

In [ ]:
final_model = search.best_estimator_.fit(X, y)
test_pred = final_model.predict(X_test)
submission = pd.DataFrame({"passenger_id": test.index, "survived": test_pred})
make_submission(submission, "titanic_submission.csv")

private_score = (test_pred == test_labels.to_numpy()).mean()
print(f"🏁 Private leaderboard accuracy: {private_score:.3f}   (CV said {search.best_score_:.3f})")
check(abs(private_score - search.best_score_) < 0.05,
      "Your CV estimate was trustworthy: that is the real skill of a competitor!",
      "Big gap between CV and test: re-check for leakage or randomness.")

## 9. 🔮 Would **you** have survived?

What did the model learn? First, the most important features. Then, create a passenger who looks like you.

In [ ]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(final_model, X, y, n_repeats=10, random_state=SEED)
pd.Series(imp.importances_mean, index=X.columns).sort_values().plot.barh(title="Permutation importance (accuracy drop)")
plt.show()

In [ ]:
me = pd.DataFrame([{
    "age": 28, "fare": 15.0, "sibsp": 0, "parch": 0, "family_size": 1, "is_alone": 1, "pclass": 3,   # ✏️ edit me!
    "sex": "male", "embarked": "S", "title": "Mr", "deck": "U",
}])
p = final_model.predict_proba(me[num_cols + cat_cols])[0, 1]
print(f"🛟 Probability of survival: {p:.0%}  →  {'you make it to a lifeboat!' if p > 0.5 else 'the North Atlantic is cold…'}")

## 🏁 Quest complete!

You did a full competition loop: EDA → leakage hunt → features → pipeline → baselines → CV → tuning → submission.

### 🚀 Your turn (bonus XP)
1. Add a feature `fare_per_person = fare / family_size`. Does the CV improve?
2. Try `LGBMClassifier` from `lightgbm` or `CatBoostClassifier` from `catboost` (pass `cat_features`!).
3. Average the probabilities of your logistic regression and your forest (see the playbook's ensembling section).
4. Ready for the real thing? Submit on Kaggle's ["Titanic: Machine Learning from Disaster"](https://www.kaggle.com/competitions/titanic).

Next quest: `02_classification_app_telco_churn.ipynb`, a real business problem.

In [ ]:
checkpoint("titanic/complete", xp=170)
badge("Titanic Survivor", "🚢")
progress()